In [1]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [2]:
# 导入rai必要工具箱
from responsibleai import RAIInsights
from raiwidgets import ResponsibleAIDashboard

/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 加载数据集
data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [4]:
# 重要：将回归问题的连续值 (y) 转换为二分类标签 (0/1)
# 以疾病进展的中位数作为分界线
y_binary = (y > y.median()).astype(int)

In [5]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

In [6]:
# 3. 特征缩放（逻辑回归通常需要）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# 4. 训练逻辑回归模型
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [8]:
y_pred_class = model.predict(X_test)
from sklearn import metrics
print(metrics.accuracy_score(y_test, y_pred_class))

0.5280898876404494


/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [9]:
y_test.mean()

0.449438202247191

In [10]:
# 5. 准备 RAIInsights 所需的数据格式
# 需要将特征和目标变量合并，以便传入 RAIInsights
train_data = X_train.copy()
test_data = X_test.copy()
train_data['target'] = y_train.values
test_data['target'] = y_test.values

In [11]:
# 6. 初始化 RAIInsights 对象
# 关键：task_type 必须设置为 'classification'
rai_insights = RAIInsights(
    model=model,                     # 训练好的模型
    train=train_data,                # 包含特征和目标的训练集
    test=test_data,                  # 包含特征和目标的测试集
    target_column='target',         # 目标列的名称
    task_type='classification',      # 任务类型是分类[reference:2][reference:3]
)

/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:48

In [12]:
# 7. 向仪表板添加核心组件
# 添加可解释性（Interpretability）和错误分析（Error Analysis）组件[reference:4]
rai_insights.explainer.add()
rai_insights.error_analysis.add()

In [13]:
# 8. 计算所有组件的洞察数据
# 这一步会运行后台计算，生成仪表板所需的所有数据[reference:5]
rai_insights.compute()

/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:48

Causal Effects
Current Status: Generating Causal Effects.
Current Status: Finished generating causal effects.
Time taken: 0.0 min 6.604100053664297e-05 sec
Counterfactual
Time taken: 0.0 min 7.167000148911029e-06 sec
Error Analysis
Current Status: Generating error analysis reports.
Current Status: Finished generating error analysis reports.
Time taken: 0.0 min 0.0361574170019594 sec
Explanations
Current Status: Explaining 10 features
Current Status: Explained 10 features.
Time taken: 0.0 min 0.10446195799886482 sec


/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [30]:
# 9. 启动仪表板！
# 执行后，会在 Notebook 中生成一个交互式界面[reference:6]
ResponsibleAIDashboard(rai_insights, port=8000)

/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/envs/rai_use/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


ResponsibleAI started at http://localhost:8000
